# TER — Fine-tuning DeBERTa sur SNLI avec Optuna

Objectif du notebook :

1. Charger et préparer le dataset **SNLI**.
2. Chercher les meilleurs hyperparamètres de **DeBERTa** avec **Optuna** sur un sous-ensemble SNLI.
3. Réentraîner **DeBERTa** sur tout le train SNLI avec les meilleurs hyperparamètres trouvés.
4. Sauvegarder le modèle final, le tokenizer, les logs d'entraînement et les hyperparamètres.


In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
import os
import gc
import json
import random
import optuna
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
    EarlyStoppingCallback,
)

import evaluate

## 1. Configuration générale

On garde une structure comparable aux notebooks BERT et RoBERTa :

- Optuna est lancé sur un sous-ensemble de SNLI pour limiter le coût GPU ;
- le modèle final est ensuite entraîné sur l'ensemble complet du train SNLI ;
- les fichiers sont sauvegardés dans Google Drive pour être réutilisés dans le notebook d'évaluation.

In [ ]:
# ==============================
# Configuration générale
# ==============================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 128
NUM_LABELS = 3

# Labels SNLI Hugging Face : 0 = entailment, 1 = neutral, 2 = contradiction
ID2LABEL = {
    0: "entailment",
    1: "neutral",
    2: "contradiction",
}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}

# Taille utilisée uniquement pour Optuna
# Test rapide : 10_000 / 2_000 / 3 trials
# Protocole propre : 55_000 / 5_000 / 10 trials
OPTUNA_TRAIN_SIZE = 55_000
OPTUNA_VALIDATION_SIZE = 5_000
N_TRIALS = 10

# Chemins Google Drive
BASE_DIR = os.environ.get("TER_OUTPUT_DIR", ".")
FINAL_MODEL_DIR = os.path.join(BASE_DIR, "models", "final_deberta_snli_model")
OPTUNA_OUTPUT_DIR = f"{DRIVE_PATH}/optuna/deberta_snli"
LOGGING_DIR = f"{DRIVE_PATH}/logs/deberta_snli"
RESULTS_DIR = os.path.join(BASE_DIR, "logs", "deberta_snli")

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
os.makedirs(OPTUNA_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGGING_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

USE_FP16 = torch.cuda.is_available()

print("MODEL_NAME :", MODEL_NAME)
print("USE_FP16 :", USE_FP16)
print("Dossier de sauvegarde final :", FINAL_MODEL_DIR)

## 2. Chargement du tokenizer et du dataset SNLI

On retire les lignes dont le label vaut `-1`, car elles correspondent à des exemples sans annotation exploitable.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Chargement du dataset SNLI...")
dataset = load_dataset("snli")

train_dataset_full = dataset["train"].filter(lambda x: x["label"] != -1)
validation_dataset_full = dataset["validation"].filter(lambda x: x["label"] != -1)

def afficher_tailles():
    print("Train complet :", len(train_dataset_full))
    print("Validation complète :", len(validation_dataset_full))

afficher_tailles()

## 3. Création des datasets pour Optuna et pour l'entraînement final

On sépare clairement :

- `optuna_train_dataset` et `optuna_validation_dataset` : utilisés uniquement pour la recherche d'hyperparamètres ;
- `final_train_dataset` et `final_validation_dataset` : utilisés pour l'entraînement final et le suivi de la perte de validation.

In [ ]:
train_dataset_shuffled = train_dataset_full.shuffle(seed=SEED)
validation_dataset_shuffled = validation_dataset_full.shuffle(seed=SEED)

optuna_train_dataset = train_dataset_shuffled.select(
    range(min(OPTUNA_TRAIN_SIZE, len(train_dataset_shuffled)))
)
optuna_validation_dataset = validation_dataset_shuffled.select(
    range(min(OPTUNA_VALIDATION_SIZE, len(validation_dataset_shuffled)))
)

final_train_dataset = train_dataset_full
final_validation_dataset = validation_dataset_full

print("Optuna train :", len(optuna_train_dataset))
print("Optuna validation :", len(optuna_validation_dataset))
print("Final train :", len(final_train_dataset))
print("Final validation :", len(final_validation_dataset))

## 4. Tokenisation DeBERTa

DeBERTa reçoit directement la paire `(premise, hypothesis)`.  
On fixe `max_length=128` afin de rester comparable avec les entraînements BERT et RoBERTa.

In [ ]:
def preprocess_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

columns_to_remove = ["premise", "hypothesis"]

print("Tokenisation des datasets Optuna...")
tokenized_optuna_train = optuna_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)
tokenized_optuna_validation = optuna_validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

print("Tokenisation des datasets finaux...")
tokenized_final_train = final_train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)
tokenized_final_validation = final_validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=columns_to_remove,
)

tokenized_optuna_train.set_format("torch")
tokenized_optuna_validation.set_format("torch")
tokenized_final_train.set_format("torch")
tokenized_final_validation.set_format("torch")

print(tokenized_final_train)

## 5. Métriques utilisées pendant Optuna

La métrique principale utilisée pour choisir les hyperparamètres est la **macro-F1**, car elle donne le même poids aux trois classes NLI.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels,
    )["accuracy"]

    macro_f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro",
    )["f1"]

    weighted_f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted",
    )["f1"]

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }

## 6. Recherche d'hyperparamètres avec Optuna

On cherche les principaux hyperparamètres utiles au fine-tuning :

- learning rate ;
- weight decay ;
- nombre d'époques ;
- batch size ;
- warmup ratio.

Le but n'est pas de faire une recherche infinie, mais de garder la même logique expérimentale que pour BERT et RoBERTa.

In [ ]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )


def objective(trial):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)
    num_train_epochs = trial.suggest_int("num_train_epochs", 2, 4)
    per_device_train_batch_size = trial.suggest_categorical(
        "per_device_train_batch_size", [8, 16]
    )
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"{OPTUNA_OUTPUT_DIR}/trial_{trial.number}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        report_to="none",
        fp16=False,

        learning_rate=learning_rate,
        per_device_train_batch_size=per_device_train_batch_size,
        per_device_eval_batch_size=32,
        num_train_epochs=num_train_epochs,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,

        seed=SEED,
        load_best_model_at_end=False,
    )

    trainer = Trainer(
        model_init=model_init,
        args=args,
        train_dataset=tokenized_optuna_train,
        eval_dataset=tokenized_optuna_validation,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    macro_f1 = eval_results["eval_macro_f1"]

    del trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return macro_f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS)

print("Meilleurs hyperparamètres :")
print(study.best_params)
print("Meilleur score macro-F1 Optuna :", study.best_value)

# Sauvegarde des résultats Optuna
optuna_results_path = f"{RESULTS_DIR}/optuna_deberta_results.csv"
study.trials_dataframe().to_csv(optuna_results_path, index=False)

best_params_path = f"{RESULTS_DIR}/best_hyperparameters_deberta.json"
with open(best_params_path, "w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "best_params": study.best_params,
        "best_macro_f1": study.best_value,
        "n_trials": N_TRIALS,
        "optuna_train_size": len(optuna_train_dataset),
        "optuna_validation_size": len(optuna_validation_dataset),
    }, f, indent=4)

print("Résultats Optuna sauvegardés dans :", optuna_results_path)
print("Meilleurs hyperparamètres sauvegardés dans :", best_params_path)

## 7. Récupération des meilleurs hyperparamètres

Cette cellule permet aussi de relancer uniquement l'entraînement final plus tard, à condition de renseigner manuellement `BEST_PARAMS` avec les valeurs déjà sauvegardées.

In [ ]:
BEST_PARAMS = study.best_params.copy()

# Valeurs de sécurité si une clé manque
BEST_PARAMS.setdefault("learning_rate", 2e-5)
BEST_PARAMS.setdefault("per_device_train_batch_size", 16)
BEST_PARAMS.setdefault("num_train_epochs", 3)
BEST_PARAMS.setdefault("weight_decay", 0.01)
BEST_PARAMS.setdefault("warmup_ratio", 0.06)

print("BEST_PARAMS utilisés pour l'entraînement final :")
print(json.dumps(BEST_PARAMS, indent=4))

## 8. Nettoyage mémoire avant l'entraînement final

On libère la mémoire GPU avant de repartir sur un modèle DeBERTa neuf pour l'entraînement final.

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Mémoire nettoyée. Prêt pour l'entraînement final DeBERTa.")

## 9. Entraînement final de DeBERTa sur tout le train SNLI

Le modèle final est entraîné sur tout le train SNLI avec les meilleurs hyperparamètres trouvés par Optuna.  
La validation SNLI sert uniquement à suivre l'apprentissage et à sélectionner le meilleur checkpoint.

In [ ]:
final_training_args = TrainingArguments(
    output_dir=FINAL_MODEL_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    logging_dir=LOGGING_DIR,
    report_to="none",
    fp16=USE_FP16,

    learning_rate=BEST_PARAMS["learning_rate"],
    per_device_train_batch_size=BEST_PARAMS["per_device_train_batch_size"],
    per_device_eval_batch_size=32,
    num_train_epochs=BEST_PARAMS["num_train_epochs"],
    weight_decay=BEST_PARAMS["weight_decay"],
    warmup_ratio=BEST_PARAMS["warmup_ratio"],

    seed=SEED,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    torch_dtype=torch.float32
)

final_model = final_model.float()

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_final_train,
    eval_dataset=tokenized_final_validation,
    compute_metrics=compute_metrics,
)

final_trainer.train()

## 10. Sauvegarde du modèle final

Le modèle sauvegardé ici sera ensuite utilisé dans le notebook d'évaluation commun avec BERT et RoBERTa.

In [ ]:
# Sauvegarde du meilleur modèle et du tokenizer
final_trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

# Sauvegarde des logs d'entraînement
log_history = pd.DataFrame(final_trainer.state.log_history)
training_log_path = f"{RESULTS_DIR}/training_log_deberta.csv"
log_history.to_csv(training_log_path, index=False)

# Sauvegarde d'un résumé d'entraînement
summary = {
    "model": "DeBERTa",
    "model_name": MODEL_NAME,
    "final_train_size": len(final_train_dataset),
    "final_validation_size": len(final_validation_dataset),
    "optuna_train_size": len(optuna_train_dataset),
    "optuna_validation_size": len(optuna_validation_dataset),
    "n_trials": N_TRIALS,
    "best_hyperparameters": BEST_PARAMS,
    "optuna_best_macro_f1": study.best_value,
    "final_model_dir": FINAL_MODEL_DIR,
}

summary_path = f"{RESULTS_DIR}/training_summary_deberta.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=4)

print("Modèle final sauvegardé dans :", FINAL_MODEL_DIR)
print("Logs sauvegardés dans :", training_log_path)
print("Résumé sauvegardé dans :", summary_path)

## 11. Courbes de suivi de l'entraînement

Ces courbes servent uniquement à contrôler que l'entraînement s'est déroulé correctement.  
L'évaluation comparative complète reste dans le notebook dédié.

In [ ]:
log_history = pd.DataFrame(final_trainer.state.log_history)
log_history.head()

In [ ]:
# Courbe de training loss
train_loss_df = log_history.dropna(subset=["loss"])[["epoch", "loss"]]

plt.figure(figsize=(8, 5))
plt.plot(train_loss_df["epoch"], train_loss_df["loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("DeBERTa — Évolution de la training loss")
plt.grid(True)
plt.savefig(f"{RESULTS_DIR}/deberta_training_loss.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Courbe de validation loss
validation_loss_df = log_history.dropna(subset=["eval_loss"])[["epoch", "eval_loss"]]

plt.figure(figsize=(8, 5))
plt.plot(validation_loss_df["epoch"], validation_loss_df["eval_loss"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("DeBERTa — Évolution de la validation loss")
plt.grid(True)
plt.savefig(f"{RESULTS_DIR}/deberta_validation_loss.png", dpi=200, bbox_inches="tight")
plt.show()

## 12. Vérification des fichiers sauvegardés

À la fin, le dossier `FINAL_MODEL_DIR` doit contenir les fichiers du modèle Hugging Face, notamment `config.json`, les poids du modèle et les fichiers du tokenizer.

In [ ]:
print("Dossier modèle final :", FINAL_MODEL_DIR)
print(os.listdir(FINAL_MODEL_DIR)[:30])

print("Dossier des sorties d'entraînement :", RESULTS_DIR)
print(os.listdir(RESULTS_DIR))